# delucionqa Experiment Runner — OpenRouter variant

RAG ablation sweep for the **delucionqa** subset of `galileo-ai/ragbench`
(automotive owner's-manual QA), with all LLM calls through **OpenRouter**.
Run alongside the other OpenRouter notebooks — it uses its own
`config/`, `reports/`, `temp/`, and `cache_openrouter_delucionqa/` dirs.

Driven by `experiment_configs/delucionqa_openrouter_experiment.yaml`.
**Requires** `OPENROUTER_API_KEY` (comma-separate multiple keys to rotate).


## 1. Setup & Dependencies

In [1]:
get_ipython().system('pip3 install datasets faiss-cpu sentence-transformers torch groq openai python-dotenv nltk pandas -q')


In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive/')

# 2. Change directory to your uploaded folder
# Update 'Your_Folder_Name' to match your actual folder path in Drive
folder_path = '/content/drive/MyDrive/Capstone/rag_cust_support'
os.chdir(folder_path)

# 3. Verify files in your current working directory
print("Current Directory:", os.getcwd())
print("Files in folder:", os.listdir())

## 2. Imports

In [2]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# --- Point this at wherever THIS repo (rag_cust_support) lives. ---
# On Colab this is typically under your mounted Drive. Adjust if different.
PROJECT_ROOT = Path('/content/drive/MyDrive/Capstone/rag_cust_support')
if not PROJECT_ROOT.exists():
    # Fallback: running locally from the notebooks/ folder.
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()

os.chdir(PROJECT_ROOT)
project_root = PROJECT_ROOT
# Make THIS repo win on sys.path (avoids importing a stale rag-foundry copy).
sys.path = [p for p in sys.path if 'rag-foundry' not in p]
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))

from experiment.experiment_config import ExperimentConfig
from experiment.experiment_runner import ExperimentRunner
import experiment.experiment_runner as _er, core.registry as _reg

load_dotenv(override=True)
print('Current directory:', Path.cwd())
print('experiment_runner loaded from:', _er.__file__)
print('core.registry   loaded from:', _reg.__file__)
assert 'rag-foundry' not in _er.__file__, 'Still importing the old rag-foundry code! Restart runtime.'
print('HuggingFace token loaded:', bool(os.getenv('HF_TOKEN')))
print('Groq API key loaded:', bool(os.getenv('GROQ_API_KEY')))
print('OpenRouter API key loaded:', bool(os.getenv('OPENROUTER_API_KEY')))


Current directory: /content/drive/MyDrive/Capstone/rag_cust_support
experiment_runner loaded from: /content/drive/MyDrive/Capstone/rag_cust_support/experiment/experiment_runner.py
core.registry   loaded from: /content/drive/MyDrive/Capstone/rag_cust_support/core/registry.py
HuggingFace token loaded: True
Groq API key loaded: True
OpenRouter API key loaded: True


## 3. Load Experiment Configuration

The experiment configuration file specifies:
- **data_loader**: How to load data (HuggingFace with dataset_name, subset, split)
- **data_parser**: How to parse documents (title_passage)
- **config_dir**: Directory containing RAG pipeline configs
- **num_queries**: Number of queries to evaluate
- **parallel**: Whether to run configs in parallel

In [3]:
EXPERIMENT_CONFIG_PATH = project_root / "experiment_configs/delucionqa_openrouter_experiment.yaml"

experiment_config = ExperimentConfig.load(EXPERIMENT_CONFIG_PATH)

print("Experiment Configuration:")
print(f"  Config Dir:  {experiment_config.config_dir}")
print(f"  Report Dir:  {experiment_config.report_dir}")
print(f"  Temp Dir:    {experiment_config.temp_dir}")
print(f"  Cache:       {experiment_config.cache}")
print(f"  Num Queries: {experiment_config.end_index}")
print(f"  Parallel:    {experiment_config.parallel}")
print(f"  Max Workers: {experiment_config.max_workers}")
print(f"\nData Loader:")
print(f"  Type: {experiment_config.data_loader['type']}")
print(f"  Config: {experiment_config.data_loader['config']}")
print(f"\nData Parser:")
print(f"  Type: {experiment_config.data_parser}")

Experiment Configuration:
  Config Dir:  rag-experiments/delucionqa-openrouter-experiment/config
  Report Dir:  rag-experiments/delucionqa-openrouter-experiment/reports
  Temp Dir:    rag-experiments/delucionqa-openrouter-experiment/temp
  Cache:       {'enabled': True, 'cache_dir': './cache_openrouter_delucionqa'}
  Num Queries: 20
  Parallel:    False
  Max Workers: 1

Data Loader:
  Type: huggingface
  Config: {'dataset_name': 'galileo-ai/ragbench', 'subset': 'delucionqa', 'split': 'test', 'limit': 184}

Data Parser:
  Type: noop


## 4. Initialize Experiment Runner

The ExperimentRunner will:
- Create report directory if it doesn't exist
- Load RAG configs from the specified directory
- Load and parse data automatically based on YAML config

In [4]:
# Initialize experiment runner
runner = ExperimentRunner(experiment_config)
print("ExperimentRunner initialized")

ExperimentRunner initialized


In [5]:
# Load data automatically based on YAML configuration
print("Loading data based on experiment configuration...")
documents, raw_data = runner.load_data()

print(f"\n✅ Data loaded successfully!")
print(f"  Documents: {len(documents)} parsed documents")
print(f"  Raw Data:  {len(raw_data)} samples")

# Inspect first sample
first_sample = raw_data[0]
print(f"\nFirst Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")

Loading data based on experiment configuration...
Loading HuggingFace dataset: galileo-ai/ragbench/delucionqa (test)...
Loaded 184 samples

✅ Data loaded successfully!
  Documents: 235 parsed documents
  Raw Data:  184 samples

First Sample:
  Question: What if I fail to latch the tailgate properly?...
  Documents: 3


## 6. Load RAG Pipeline Configs

Load all RAG pipeline configurations from the config directory specified in the experiment config.

In [6]:
# Load RAG pipeline configs
configs = runner.load_configs()

print(f'Loaded {len(configs)} RAG pipeline configurations:')
for cfg in configs:
    searches = ' + '.join(s.type.value for s in cfg.retrieval.search.searches)
    fusion = cfg.retrieval.fusion.type.value if cfg.retrieval.fusion else '-'
    rerank = cfg.retrieval.rerank.type.value if cfg.retrieval.rerank else '-'
    qx = cfg.retrieval.query_transform.type.value if cfg.retrieval.query_transform else '-'
    gc = cfg.generation.config
    model = gc.get('model') if isinstance(gc, dict) else getattr(gc, 'model', None)
    print(f'  - {cfg.name}')
    print(f'      chunking={cfg.chunking.type.value}  embed={cfg.embedding.type.value}')
    print(f'      search=[{searches}]  fusion={fusion}  rerank={rerank}  q_transform={qx}')
    print(f'      generation_model={model}')


Loaded 8 RAG pipeline configurations:
  - delucionqa_or_v1_baseline
      chunking=fixed_word  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=meta-llama/llama-3.3-70b-instruct
  - delucionqa_or_v2_hybrid_wsum
      chunking=fixed_word  embed=sentence_transformer
      search=[dense + sparse]  fusion=weighted_sum  rerank=-  q_transform=-
      generation_model=meta-llama/llama-3.3-70b-instruct
  - delucionqa_or_v3_embed_bge
      chunking=fixed_word  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=meta-llama/llama-3.3-70b-instruct
  - delucionqa_or_v4_chunk_sentence
      chunking=sentence  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=meta-llama/llama-3.3-70b-instruct
  - delucionqa_or_v5_hybrid_rrf
      chunking=fixed_word  embed=sentence_transformer
      search=[dense + sparse]  fusion=rrf  rerank=-  q_trans

## 7. Run Experiments

Run all RAG configurations on the loaded data. Each config will:
1. Build a vector index from the documents
2. Run queries against the index
3. Generate responses
4. Evaluate using TRACe metrics

Results are returned as PipelineRunResult objects.

In [7]:
get_ipython().system('pip install rank_bm25 -q')

# Run experiments

In [8]:
# Run experiments

print(f"Running {len(configs)} configurations from {experiment_config.start_index} to {experiment_config.end_index} queries...")
print(f"Parallel mode: {experiment_config.parallel}")

runs = runner.run(documents, raw_data)

print(f"\n✅ Experiments completed!")
print(f"  Ran {len(runs)} configurations")

for run in runs:
    print(f"  - {run['config'].name}: {run['total_written']} queries")

Running 8 configurations from 0 to 20 queries...
Parallel mode: False
OpenRouter provider initialized with 4 key(s) (rotation needs 2+ keys; set OPENROUTER_API_KEY comma-separated).
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 0sUsing key #0: ****f2e4Using key #0: ****f2e4

HTTP 402 (retryable) on ****f2e4
Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402}}
{'provider': 'openrouter', 'current_index': 0, 'keys': [{'key_suffix': 'f2e4', 'available': False, 'cooldown_until': datetime.datetime(2026, 7, 25, 1, 1, 49, 991993), 'requests': 2, 'successes': 0, 'failures': 1, '429s': 1}, {'key_suffix': 'e916', 'available': True, 'cooldown_until': None, 'requests': 0, 'successes': 0, 'failures': 0, '429s': 0}, {'key_suffix': 'b076', 'available': True, 'cooldown_until': None, 'requests': 0, 'successes': 0, 'failures': 0, '429s': 0}, {'key_suffix': 'fbdb', 'available': True, 'cooldown_until': None, 'requests': 0, 'successes': 0, 'failures': 0, '429s': 0}]}
Using key #1: ****e916
H

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 1.8mUsing key #1: ****e916
Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 1.9mUsing key #1: ****e916
Progress: 2/20 (10.0%) | QPS: 0.01 | ETA: 34.5m | Elapsed: 3.8mUsing key #1: ****e916
Using key #1: ****e916
Progress: 4/20 (20.0%) | QPS: 0.01 | ETA: 21.9m | Elapsed: 5.5mUsing key #1: ****e916
Progress: 5/20 (25.0%) | QPS: 0.01 | ETA: 17.3m | Elapsed: 5.8mUsing key #1: ****e916
Progress: 6/20 (30.0%) | QPS: 0.01 | ETA: 17.6m | Elapsed: 7.6mUsing key #1: ****e916
Progress: 7/20 (35.0%) | QPS: 0.02 | ETA: 14.2m | Elapsed: 7.6mUsing key #1: ****e916
Progress: 8/20 (40.0%) | QPS: 0.01 | ETA: 14.2m | Elapsed: 9.5mUsing key #1: ****e916
Using key #1: ****e916
Progress: 10/20 (50.0%) | QPS: 0.01 | ETA: 11.4m | Elapsed: 11.4mUsing key #1: ****e916
Progress: 11/20 (55.0%) | QPS: 0.02 | ETA: 9.6m | Elapsed: 11.7mUsing key #1: ****e916
Progress: 12/20 (60.0%) | QPS: 0.02 | ETA: 8.8m | Elapsed: 13.3mUsing key #1: ****e916
Pr

## 7b. Evaluate Existing JSONL Files

Run offline evaluation on already-generated JSONL files.
Uses experiment-level evaluation config — all configs are scored with the same judge model.

- `parallel_runs=True` — evaluate multiple configs simultaneously
- `parallel_config_run=True` — evaluate records within each config in parallel

In [9]:
# Discover all configs and build run dicts from existing JSONL files
configs = runner.load_configs()
runs = []
for cfg in configs:
    jsonl_path = experiment_config.temp_dir / f"{cfg.name}.jsonl"
    if jsonl_path.exists():
        runs.append({"config_name": cfg.name, "config": cfg, "jsonl_path": jsonl_path})
    else:
        print(f"  Skipping {cfg.name} — no JSONL found")

print(f"Found {len(runs)} configs with JSONL files")

# Evaluate all configs: parallel across configs + parallel within each config
eval_runs = runner.evaluate_runs(
    runs,
    parallel_runs=True,
    parallel_config_run=True,
)

# Use eval_runs for report generation downstream
runs = eval_runs
print(f"\n✅ Evaluation complete: {len(eval_runs)} configs")

Found 8 configs with JSONL files


Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
[delucionqa_or_v1_baseline] Evaluation complete → rag-experiments/delucionqa-openrouter-experiment/temp/delucionqa_or_v1_baseline.jsonl
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916


Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916



```
{
  "relevance_explanation": "The question asks about using the key fob to unlock all the doors. Relevant sentences are those that describe the key fob's unlock functionality, particularly in relation to unlocking all doors.",
  "all_relevant_sentence_keys": ["d0s0", "d0s


[delucionqa_or_v2_hybrid_wsum] Evaluation complete → rag-experiments/delucionqa-openrouter-experiment/temp/delucionqa_or_v2_hybrid_wsum.jsonl
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
[delucionqa_or_v3_embed_bge] Evaluation complete → rag-experiments/delucionqa-openrouter-experiment/temp/delucionqa_or_v3_embed_bge.jsonl
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916


...

Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
[delucionqa_or_v5_hybrid_rrf] Evaluation complete → rag-experiments/delucionqa-openrouter-experiment/temp/delucionqa_or_v5_hybrid_rrf.jsonl
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916


Using key #1: ****e916


Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
[delucionqa_or_v6_rerank_only] Evaluation complete → rag-experiments/delucionqa-openrouter-experiment/temp/delucionqa_or_v6_rerank_only.jsonl
Using key #1: ****e916
Using key #1: ****e916



```
{
  "relevance_explanation": "The question asks about the consequences of failing to latch the tailgate properly. The relevant sentences in the documents discuss the importance of securely latching the tailgate and the potential damage that could result from failure to do so.",
  "all_relevant_sentence_keys": ["d0s1", "d0s2", "d1s1", "d1s


Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
[delucionqa_or_v7_hybrid_rerank] Evaluation complete → rag-experiments/delucionqa-openrouter-experiment/temp/delucionqa_or_v7_hybrid_rerank.jsonl
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Using key #1: ****e916
Usi

## 8. Generate Reports

Generate detailed reports for each configuration including:
- Per-query table with all TRACe scores
- Aggregate statistics (mean, std, MAE)
- Comparison with ground truth

In [10]:
# Generate reports
print("Generating reports...")
reports = runner.generate_reports(runs)

print(f"\n✅ Reports generated!")
print(f"  Saved to: {experiment_config.report_dir}")

Generating reports...

✅ Reports generated!
  Saved to: rag-experiments/delucionqa-openrouter-experiment/reports


## 9. Display Reports

Display the generated reports with per-query and aggregate metrics.

In [11]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")

    # Display per-query results
    print("\nPer-Query Results:")
    display(report.display())



Configuration: delucionqa_or_v1_baseline

Per-Query Results:


# RAG Multi-Config Evaluation Report

_Strategy: detailed_query_

## Config: `delucionqa_or_v1_baseline`

**name**: delucionqa_or_v1_baseline  •  **mode**: test  •  **providers**: {'openrouter': {'type': 'openrouter', 'api_key_env': 'OPENROUTER_API_KEY', 'params': {'cooldown_seconds': 60}}}  •  **chunking**: {'type': 'fixed_word', 'config': {'max_words': 160, 'overlap_words': 24}}  •  **embedding**: {'type': 'sentence_transformer', 'config': {'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'dimension': 384}}  •  **vector_store**: {'type': 'faiss', 'config': {'dimension': 384}}  •  **retrieval**: {'search': {'searches': [{'type': 'dense', 'config': {'top_k': 5}}]}, 'query_transform': None, 'fusion': None, 'rerank': None}  •  **generation**: {'strategy': 'default', 'provider': 'openrouter', 'config': {'model': 'meta-llama/llama-3.3-70b-instruct', 'temperature': 0.0, 'max_tokens': 512, 'system_prompt': 'You are an automotive owner\'s-manual support assistant. Answer questions\nabout vehicle features, controls, and procedures using ONLY the provided\nowner\'s-manual passages.\n\nCRITICAL RULES:\n1. Answer ONLY from the passages provided. Do not use outside knowledge.\n2. Give exact, step-by-step procedures. Preserve exact control names, button\n   names, warning/CAUTION text, and feature labels verbatim.\n3. Include any safety warnings (WARNING/CAUTION) that the passages attach to\n   the procedure.\n4. Be concise and procedural - give the steps, not background.\n5. If the passages do not contain the answer, respond with exactly:\n   "The passages do not provide sufficient information to answer this question."\n6. Every claim must be directly supported by the passages provided.\n', 'user_prompt': 'Passages:\n{context}\n\nQuestion: {query}\n\nAnswer (from passages only, give exact steps, control names, and any safety warnings):\n'}}  •  **evaluation**: {'type': 'trace', 'provider': 'openrouter', 'config': {'model': 'meta-llama/llama-3.3-70b-instruct', 'temperature': 0.0, 'max_tokens': 2000}}  •  **cache**: {'enabled': True, 'cache_dir': './cache_openrouter_delucionqa'}  •  **start_index**: None  •  **end_index**: None  •  **logging_config**: {'enabled': True, 'level': 'INFO', 'show_progress': True}

...

### Per-query results

,query,retrieved_documents,answer,relevance_score__pred,relevance_score__gt,relevance_score__deviation,utilization_score__pred,utilization_score__gt,utilization_score__deviation,completeness_score__pred,completeness_score__gt,completeness_score__deviation,adherence_score__pred,adherence_score__gt,adherence_score__deviation
0,What if I fail to latch the tailgate properly?,"[{'text': '', 'title': '', 'content': 'Closing To close the tailgate, lift u...",CAUTION: Failure to securely latch the tailgate could result in damage to th...,0.1333,0.1111,0.0222,0.1333,0.1111,0.0222,1.0000,1.0000,0.0000,1.0,1.0,0.0
1,What kind of safety features are implemented in this car?,"[{'text': '', 'title': '', 'content': 'Some of the most important safety fea...",The passages mention the following safety features: \n1. Restraint systems (...,0.6857,0.2500,0.4357,0.6571,0.2500,0.4071,0.6667,1.0000,-0.3333,0.0,1.0,-1.0
2,When will the Automatic SOS be triggered?,"[{'text': '', 'title': '', 'content': 'Automatic SOS — If Equipped Automatic...",The passages do not provide sufficient information to answer this question.,0.0333,0.0244,0.0089,0.0000,0.0244,-0.0244,0.0000,1.0000,-1.0000,0.0,1.0,-1.0
3,What happens if I accidentally push the SOS Call button?,"[{'text': '', 'title': '', 'content': 'Connected Services SOS FAQs — If Equi...","If you accidentally push the SOS Call button, you have 10 seconds to cancel ...",0.2143,0.0820,0.1323,0.2143,0.0656,0.1487,0.4444,0.8000,-0.3556,0.0,1.0,-1.0
4,What is the DEF?,"[{'text': '', 'title': '', 'content': 'System Overview This vehicle is equip...","The passages do not provide a direct definition of DEF. However, according t...",0.4000,0.0625,0.3375,0.1333,0.0625,0.0708,0.2500,1.0000,-0.7500,0.0,1.0,-1.0
5,What may cause erratic or noisy performance of the radio?,"[{'text': '', 'title': '', 'content': 'Under certain conditions, the mobile ...",The mobile phone being on in your vehicle can cause erratic or noisy perform...,0.1778,0.1143,0.0635,0.0444,0.1143,-0.0699,0.2500,1.0000,-0.7500,0.0,1.0,-1.0
6,how to calculate the gross trailer weight?,"[{'text': '', 'title': '', 'content': 'Gross Trailer Weight (GTW) The GTW is...","To calculate the Gross Trailer Weight (GTW), follow these steps:\n\n1. Put y...",0.1579,0.2727,-0.1148,0.2105,0.2727,-0.0622,1.0000,1.0000,0.0000,0.0,1.0,-1.0
7,What can the ASIST button do?,"[{'text': '', 'title': '', 'content': 'ASSIST Call The ASSIST button is used...",The ASSIST button is used to automatically connect you to any one of the fol...,0.4545,0.4348,0.0197,0.3333,0.4348,-0.1015,0.7333,1.0000,-0.2667,0.0,1.0,-1.0
8,What does the Door Off Mirror Kit do?,"[{'text': '', 'title': '', 'content': 'Door Off Mirror Kit — If Equipped If ...",The Door Off Mirror Kit allows exterior rearview mirrors to be installed ont...,0.3077,0.1304,0.1773,0.3462,0.1304,0.2158,0.8750,1.0000,-0.1250,0.0,1.0,-1.0
9,Can I manually activate or deactivate the air conditioner?,"[{'text': '', 'title': '', 'content': 'A/C Button Press and release this but...","To manually activate or deactivate the air conditioner, press and release th...",0.3611,0.0811,0.2800,0.1944,0.1351,0.0593,0.4615,0.6667,-0.2052,0.0,1.0,-1.0


### Aggregate TRACe scores (mean / ground truth / deviation)

,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,relevance_score,0.3707,0.2060,0.2305,0.1651,0.1976
1,utilization_score,0.2692,0.1892,0.1834,0.1589,0.1518
2,completeness_score,0.5349,0.8762,0.3263,0.2242,0.3867
3,adherence_score,0.2500,1.0000,0.4330,0.0000,0.7500


None

## 10. Compare Configurations

Generate a comparison report across all configurations to see which performs best.

In [12]:
# Generate comparison report
print("Generating comparison report...")
comparison = runner.compare()

print(f"\n✅ Comparison report generated!")
print(f"  Saved to: {experiment_config.report_dir}/comparison.csv")

Generating comparison report...

✅ Comparison report generated!
  Saved to: rag-experiments/delucionqa-openrouter-experiment/reports/comparison.csv


In [13]:
# Display comparison
print("\nConfiguration Comparison:")
display(comparison.to_dataframe())


Configuration Comparison:


,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,delucionqa_or_v1_baseline,0.2979,0.1954,0.2374,0.1781,0.6330,0.3524,0.1579,0.8421
1,delucionqa_or_v2_hybrid_wsum,0.1780,0.1498,0.1507,0.1250,0.6429,0.3488,0.1111,0.8889
2,delucionqa_or_v3_embed_bge,0.3530,0.2279,0.2953,0.1781,0.6375,0.3120,0.2500,0.7500
3,delucionqa_or_v4_chunk_sentence,0.4214,0.2532,0.2906,0.1721,0.5869,0.3720,0.2105,0.7895
4,delucionqa_or_v5_hybrid_rrf,0.1543,0.1245,0.1488,0.1052,0.7612,0.2692,0.1111,0.8889
5,delucionqa_or_v6_rerank_only,0.3484,0.1554,0.2844,0.1597,0.6007,0.3483,0.2222,0.7778
6,delucionqa_or_v7_hybrid_rerank,0.3708,0.1930,0.2689,0.1353,0.6322,0.2889,0.2105,0.7895
7,delucionqa_or_v8_hyde,0.3707,0.1976,0.2692,0.1518,0.5349,0.3867,0.2500,0.7500


## 11. Summary

The ExperimentRunner provides a complete workflow for:

1. **Configuration-driven data loading** - Specify data source in YAML
2. **Automatic parsing** - Documents parsed using configured parser
3. **Multi-config evaluation** - Test multiple RAG configurations
4. **Parallel execution** - Speed up evaluation with parallel runs
5. **Comprehensive reporting** - Per-query and aggregate metrics
6. **Cross-config comparison** - Identify best performing config

### Key Benefits:

- **Reproducible** - Everything configured in YAML
- **Flexible** - Easy to change data source or parser
- **Scalable** - Parallel execution for faster evaluation
- **Comprehensive** - Detailed metrics and comparisons